# mtDNA-FM: Haplogroup Fine-tuning on Colab T4 GPU

**Before running this notebook:**
1. Runtime → Change runtime type → T4 GPU
2. Upload `data/processed/train.parquet`, `val.parquet`, `test.parquet` to
   `My Drive/mtdna_fm_data/` in Google Drive

These parquet files were created locally by running:
```bash
mtdna-preprocess --hmtdb data/hmtdb_labeled --output-dir data/processed
python paper/experiments/evaluation/fix_haplogroup_labels.py
```

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2: Clone repo and install dependencies
import os
if not os.path.exists('mtdna-foundation-model'):
    !git clone https://github.com/vthawfeek/mtdna-foundation-model
%cd mtdna-foundation-model
!pip install -e '.[train]' -q

In [ ]:
# Cell 3: Mount Google Drive and copy processed parquets
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/mtdna_fm_data')
local_data = Path('data/processed')
local_data.mkdir(parents=True, exist_ok=True)

for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    src = drive_data / fname
    dst = local_data / fname
    if not src.exists():
        raise FileNotFoundError(
            f'{src} not found. Upload the three parquet files to '
            'My Drive/mtdna_fm_data/ first.'
        )
    shutil.copy(src, dst)
    import pandas as pd
    df = pd.read_parquet(dst)
    print(f'{fname}: {len(df):,} rows, columns: {list(df.columns)}')

print('Data ready.')

In [ ]:
# Cell 4: Quick data sanity check
import pandas as pd

train_df = pd.read_parquet('data/processed/train.parquet')
print('Train major_haplogroup distribution:')
print(train_df['major_haplogroup'].value_counts().sort_index().to_string())
print(f'\nTotal train sequences with labels: {train_df["major_haplogroup"].notna().sum():,}')

In [ ]:
# Cell 5: Run fine-tuning
# Uses configs/finetuning_haplogroup_gpu.yaml:
#   - base_model: vthawfeek/mtdna-foundation-model (HF Hub)
#   - batch_size: 64, grad_accum: 2, effective batch: 128
#   - 40 epochs, lr=1e-3, LoRA r=8
#   - Output: models/finetune_haplogroup_gpu/

!mtdna-finetune --task haplogroup --config configs/finetuning_haplogroup_gpu.yaml 2>&1 | tee finetune_log.txt

In [ ]:
# Cell 6: Check training summary
import json

metrics_path = 'models/finetune_haplogroup_gpu/eval_metrics.json'
if not Path(metrics_path).exists():
    # Check best/ subdirectory
    metrics_path = 'models/finetune_haplogroup_gpu/best/eval_metrics.json'

with open(metrics_path) as f:
    metrics = json.load(f)

print('Training complete.')
print(f'Best val accuracy: {metrics["best_val_accuracy"]:.1%}')
print(f'Final train loss:  {metrics["final_train_loss"]:.4f}')

In [ ]:
# Cell 7: Evaluate best checkpoint on held-out test set
!python paper/evaluate_haplogroup_finetune.py \
    --checkpoint models/finetune_haplogroup_gpu/best \
    --test-parquet data/processed/test.parquet \
    --output reports/finetune_haplogroup_gpu.json

In [ ]:
# Cell 8: Print results summary for paper
import json

with open('reports/finetune_haplogroup_gpu.json') as f:
    r = json.load(f)

print('=' * 50)
print('RESULTS FOR PAPER')
print('=' * 50)
print(f'Test accuracy  : {r["test_accuracy"]:.1%}')
print(f'Macro-F1       : {r["macro_f1"]:.4f}')
print(f'Random baseline: {r["random_baseline"]:.2%}')
print(f'Lift           : {r["lift"]}x')
print(f'Test windows   : {r["n_test_windows"]:,}')
print()
print('Per-class F1:')
for cls, f1 in sorted(r['per_class_f1'].items(), key=lambda x: -x[1]):
    n = int(r['per_class_n'].get(cls, 0))
    print(f'  {cls:<4}  F1={f1:.3f}  n={n}')

In [ ]:
# Cell 9: Download results JSON to your local machine
# This file is what you need to update the paper.
from google.colab import files
files.download('reports/finetune_haplogroup_gpu.json')
files.download('finetune_log.txt')

## After downloading the JSON

Place `finetune_haplogroup_gpu.json` in `reports/` in the local repo, then run:

```bash
python paper/update_paper_finetune.py
```

This updates Table 1, §4.1, the abstract, and the conclusion in `main.tex`,
adds Supplementary Table S3.2, and recompiles the PDFs.

Expected results (based on literature for BERT LoRA fine-tuning with 11k examples):
- Test accuracy: **70–85%**
- This beats the k-mer + LR baseline (~65%) and is 18–22× the random baseline (3.85%)